**Importing Libraries**

In [ ]:
import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, metrics
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

In [ ]:
import cv2

In [ ]:
from tensorflow.keras import regularizers

In [ ]:
from tensorflow.keras.layers import Input, Add, Dense, Activation, ZeroPadding2D, BatchNormalization, Flatten, Conv2D, AveragePooling2D, MaxPooling2D, GlobalMaxPooling2D, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.initializers import glorot_uniform

In [ ]:
from sklearn.metrics import confusion_matrix

**Mounting Drive for Dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount = True)

In [ ]:
# Path of the file
zipfilename = "/content/gdrive/MyDrive/FractureData-ClassificationSet-BalancedValidationTest-posnegclass-balancedvaltest.zip"

# Target directory
extract_dir = "/content/"

# Unzip the file
shutil.unpack_archive(zipfilename, "/content/")

**Declaring Dataset Paths**

In [ ]:
val_dir    = os.path.join(base_dir, "valid")

In [ ]:
base_dir_test_balanced = os.path.join(base_dir, "test")

In [ ]:
fracture_dir = os.path.join(train_dir, "fracture")
nofrac_dir   = os.path.join(train_dir, "no-fracture")

In [ ]:
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 32
EPOCHS = 300

**Data Generators**

In [ ]:
# Validation generator
val_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
val_gen = val_datagen.flow_from_directory(
    val_dir,
    classes=['no-fracture', 'fracture'],
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode='grayscale',
    class_mode='binary',
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
# Testing generator
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
test_gen = test_datagen.flow_from_directory(
    base_dir_test_balanced,
    classes=['no-fracture', 'fracture'],
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode='grayscale',
    class_mode='binary',
    batch_size=BATCH_SIZE,
    shuffle=False
)

**VGG-16 Gated Multiplication Model**

In [ ]:
def build_vgg16_gated_multiplication(input_shape=(224, 224, 1), n_classes=1):
  L2 = regularizers.l2(1e-3)
  inp = layers.Input(shape=input_shape, name='input')

  x = layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv1', kernel_regularizer=L2)(inp)
  x = layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv2', kernel_regularizer=L2)(x)
  x = layers.MaxPooling2D((2, 2), strides=2, name='block1_pool')(x)

  x = layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv1', kernel_regularizer=L2)(x)
  x = layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv2', kernel_regularizer=L2)(x)
  x = layers.MaxPooling2D((2, 2), strides=2, name='block2_pool')(x)

  x = layers.Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv1', kernel_regularizer=L2)(x)
  x = layers.Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv2', kernel_regularizer=L2)(x)
  x = layers.Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv3', kernel_regularizer=L2)(x)
  x = layers.MaxPooling2D((2, 2), strides=2, name='block3_pool')(x)

  x = layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv1', kernel_regularizer=L2)(x)
  x = layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv2', kernel_regularizer=L2)(x)
  x = layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv3', kernel_regularizer=L2)(x)
  x = layers.SpatialDropout2D(0.1, name='spatial_drop_4')(x)
  x = layers.MaxPooling2D((2, 2), strides=2, name='block4_pool')(x)

  x = layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv1', kernel_regularizer=L2)(x)
  x = layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv2', kernel_regularizer=L2)(x)
  x = layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv3', kernel_regularizer=L2)(x)
  x = layers.SpatialDropout2D(0.1, name='spatial_drop_5')(x)
  x = layers.MaxPooling2D((2, 2), strides=2, name='block5_pool')(x)


  gmp_raw = layers.GlobalMaxPooling2D(name='gmp_raw')(x)
  gap_raw = layers.GlobalAveragePooling2D(name='gap_raw')(x)
  gmp_proj = layers.Dense(512, activation='relu', name='gmp_project', kernel_regularizer=L2)(gmp_raw)
  gate = layers.Dense(512, activation='sigmoid', name='attention_gate', kernel_regularizer=L2)(gap_raw)
  gated_features = layers.Multiply(name='gated_pooling')([gmp_proj, gate])
  x = layers.Dropout(0.5, name='head_dropout')(gated_features)
  out = layers.Dense(n_classes, activation='sigmoid', name='predictions', kernel_regularizer=L2)(x)

**Loading Saved Model from .h5 file**

In [ ]:
#VGG-16 with Gated Multiplication
modelll_vgg16_b32_sgdlrscheduling_valaccreccallback_newaug_gmpgapgatedspatialadjusted_dense512_classweightsthreezero_balvaltest_dppointfive_trfs = load_model("/content/gdrive/MyDrive/modelll_vgg16_b32_sgdlrscheduling_valaccreccallback_newaug_gmpgapgatedspatialadjusted_dense512_classweightsthreezero_balvaltest_dppointfive_trfs.h5")

**Load Model History Variable**

In [ ]:
with open("/kaggle/working/history_modelll_vgg16_b32_sgdlrscheduling_valaccreccallback_newaug_gmpgapgatedspatialadjusted_dense512_classweightsthreezero_balvaltest_dppointfive_trfs.pkl", "rb") as f:
    history_dict = pickle.load(f)

**Accuracy and Loss Graphs**

In [ ]:
plt.figure()
plt.plot(history_dict['loss'], label='train loss')
plt.plot(history_dict['val_loss'], label='val loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.show()

plt.figure()
plt.plot(history_dict['accuracy'], label='train acc')
plt.plot(history_dict['val_accuracy'], label='val acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.show()

**VGG with Gated Multiplication - Results Validation Set**

In [ ]:
val_preds_gated = modelll_vgg16_b32_sgdlrscheduling_valaccreccallback_newaug_gmpgapgatedspatialadjusted_dense512_classweightsthreezero_balvaltest_dppointfive_trfs.predict(val_gen, steps=int(np.ceil(val_gen.samples / BATCH_SIZE)))

# Step 2: Convert to class labels
val_pred_labels_gated = (val_preds_gated > 0.5).astype(int).flatten()

# Step 3: True labels
true_labels_gated = val_gen.classes

# Step 4: Count correct and incorrect
correct_gated = np.sum(val_pred_labels_gated == true_labels_gated)
incorrect_gated = np.sum(val_pred_labels_gated != true_labels_gated)

print(f"Correct predictions: {correct_gated}")
print(f"Incorrect predictions: {incorrect_gated}")
print(f"Validation Accuracy: {correct_gated / len(true_labels_gated):.2%}")


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


43/43 ━━━━━━━━━━━━━━━━━━━━ 27s 327ms/step
Correct predictions: 1113
Incorrect predictions: 246
Validation Accuracy: 81.90%


In [ ]:
TP_indices_gated = np.where((val_pred_labels_gated == 1) & (true_labels_gated == 1))[0]  # fracture correct
FN_indices_gated = np.where((val_pred_labels_gated == 0) & (true_labels_gated == 1))[0]  # fracture wrong
TN_indices_gated = np.where((val_pred_labels_gated == 0) & (true_labels_gated == 0))[0]  # no-fracture correct
FP_indices_gated = np.where((val_pred_labels_gated == 1) & (true_labels_gated == 0))[0]  # no-fracture wrong

print(f"Fracture Correct (TP): {len(TP_indices_gated)}")
print(f"Fracture Incorrect (FN): {len(FN_indices_gated)}")
print(f"No-Fracture Correct (TN): {len(TN_indices_gated)}")
print(f"No-Fracture Incorrect (FP): {len(FP_indices_gated)}")

Fracture Correct (TP): 228
Fracture Incorrect (FN): 27
No-Fracture Correct (TN): 885
No-Fracture Incorrect (FP): 219


**VGG-16 with Gated Spatial - Test Set Results**

In [ ]:
test_preds_gated = modelll_vgg16_b32_sgdlrscheduling_valaccreccallback_newaug_gmpgapgatedspatialadjusted_dense512_classweightsthreezero_balvaltest_dppointfive_trfs.predict(test_gen, steps=int(np.ceil(test_gen.samples / BATCH_SIZE)))

# Step 2: Convert to class labels
test_pred_labels_gated = (test_preds_gated > 0.5).astype(int).flatten()

# Step 3: True labels
test_true_labels_gated = test_gen.classes

# Step 4: Count correct and incorrect
test_correct_gated = np.sum(test_pred_labels_gated == test_true_labels_gated)
test_incorrect_gated = np.sum(test_pred_labels_gated != test_true_labels_gated)

print(f"Test Set Correct predictions: {test_correct_gated}")
print(f"Test Set Incorrect predictions: {test_incorrect_gated}")
print(f"Testing Accuracy: {test_correct_gated / len(test_true_labels_gated):.2%}")

43/43 ━━━━━━━━━━━━━━━━━━━━ 13s 296ms/step
Test Set Correct predictions: 1099
Test Set Incorrect predictions: 259
Testing Accuracy: 80.93%


In [ ]:
TP_indices_test_gated = np.where((test_pred_labels_gated == 1) & (test_true_labels_gated == 1))[0]  # fracture correct
FN_indices_test_gated = np.where((test_pred_labels_gated == 0) & (test_true_labels_gated == 1))[0]  # fracture wrong
TN_indices_test_gated = np.where((test_pred_labels_gated == 0) & (test_true_labels_gated == 0))[0]  # no-fracture correct
FP_indices_test_gated = np.where((test_pred_labels_gated == 1) & (test_true_labels_gated == 0))[0]  # no-fracture wrong

print(f"Fracture Correct (TP): {len(TP_indices_test_gated)}")
print(f"Fracture Incorrect (FN): {len(FN_indices_test_gated)}")
print(f"No-Fracture Correct (TN): {len(TN_indices_test_gated)}")
print(f"No-Fracture Incorrect (FP): {len(FP_indices_test_gated)}")

Fracture Correct (TP): 208
Fracture Incorrect (FN): 46
No-Fracture Correct (TN): 891
No-Fracture Incorrect (FP): 213


**Grad-CAM++**

In [ ]:
def make_gradcampp_heatmap(img_array, model, layer_name, pred_index=None):
    # Build a model that maps input → (conv outputs, predictions)
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape1:
        with tf.GradientTape() as tape2:
            with tf.GradientTape() as tape3:
                conv_outputs, predictions = grad_model(img_array)

                # Change 2: Explicit handling for Binary Classification (Sigmoid)
                # If the model has 1 output neuron, the index is always 0.
                if pred_index is None:
                    if predictions.shape[-1] == 1:
                        pred_index = 0
                    else:
                        pred_index = tf.argmax(predictions[0])

                epsilon = 1e-7
                output = predictions[:, pred_index]
                logits = tf.math.log(output / (1 - output + epsilon) + epsilon)

                # First derivative of LOGITS
                grads = tape3.gradient(logits, conv_outputs)

            # Second derivative
            grads2 = tape2.gradient(grads, conv_outputs)

        # Third derivative
        grads3 = tape1.gradient(grads2, conv_outputs)

    # Convert tensors to numpy-friendly format
    conv_outputs = conv_outputs[0]              # (H, W, C)
    grads = grads[0]
    grads2 = grads2[0]
    grads3 = grads3[0]

    # Compute weights (Grad-CAM++ formula)
    numerator = grads2
    denominator = 2 * grads2 + \
        tf.reduce_sum(conv_outputs * grads3, axis=(0, 1), keepdims=True)

    # Replace zeros in denominator to avoid division by zero
    alpha = numerator / (denominator + 1e-10)

    # Use ReLU on first-order gradients
    weights = tf.reduce_sum(alpha * tf.nn.relu(grads), axis=(0, 1))

    # Weighted sum of activations
    cam = tf.reduce_sum(weights * conv_outputs, axis=-1)

    # Change 3: Dead Filter Safety (Matching your Standard Grad-CAM implementation)
    heatmap = np.maximum(cam, 0)
    max_val = np.max(heatmap)

    if max_val == 0:
        return heatmap # Return empty zero map

    heatmap = heatmap / (max_val + 1e-10)

    return heatmap

# --- Get all convolutional layers ---
# Note: Using the specific model name you provided
conv_layer_names = [layer.name for layer in modelll_vgg16_b32_sgdlrscheduling_valaccreccallback_newaug_gmpgapgatedspatialadjusted_dense512_classweightsthreezero_balvaltest_dppointfive_trfs.layers if isinstance(layer, Conv2D)]
print(f"Found {len(conv_layer_names)} conv layers:", conv_layer_names)

# --- Loop through validation generator ---
for batch_index in range(len(test_gen)):
    x_batch, y_batch = test_gen[batch_index]

    for i in range(x_batch.shape[0]):
        img_array = np.expand_dims(x_batch[i], axis=0)

        # Keep original grayscale as (224,224,1)
        img_original = np.uint8(255 * x_batch[i].squeeze())  # single-channel

        # --- Get original filename ---
        file_index = batch_index * test_gen.batch_size + i
        if file_index >= len(test_gen.filenames):
            continue
        filename = os.path.basename(test_gen.filenames[file_index])
        file_base, _ = os.path.splitext(filename)

        for conv_layer_name in conv_layer_names:
            # Generate heatmap for this layer
            heatmap = make_gradcampp_heatmap(img_array, modelll_vgg16_b32_sgdlrscheduling_valaccreccallback_newaug_gmpgapgatedspatialadjusted_dense512_classweightsthreezero_balvaltest_dppointfive_trfs, conv_layer_name)

            # Resize to image size
            heatmap_resized = cv2.resize(heatmap, (x_batch[i].shape[1], x_batch[i].shape[0]))
            heatmap_resized = np.uint8(255 * heatmap_resized)

            # Apply colormap
            heatmap_colored = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)

            # Overlay (requires broadcasting grayscale → 3 channels)
            img_gray_3ch = cv2.merge([img_original]*3)
            superimposed_img = cv2.addWeighted(img_gray_3ch, 0.6, heatmap_colored, 0.4, 0)

            # Save results with original filename
            out_prefix = os.path.join(output_dir_one, f"{file_base}_{conv_layer_name}")
            cv2.imwrite(f"{out_prefix}_heatmap.png", heatmap_colored)     # heatmap
            cv2.imwrite(f"{out_prefix}_overlay.png", superimposed_img)    # overlay

    print(f"✅ Processed batch {batch_index+1}/{len(test_gen)}")

print("🎉 Done! Heatmaps saved with original filenames.")